# INS232-01 MAE CV Benchmark Pipeline

Balanced-runtime Kaggle notebook for insurance premium regression with reproducible CV, multiple model families, optional Optuna, ablations, error analysis, ensembling, and submission export.


In [ ]:
import importlib.util
import subprocess
import sys


def run_cmd(cmd):
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)


def ensure_package(module_name, package_name=None):
    pkg = package_name or module_name
    if importlib.util.find_spec(module_name) is None:
        run_cmd([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("scipy")
ensure_package("sklearn", "scikit-learn")
ensure_package("catboost")
ensure_package("lightgbm")
ensure_package("xgboost")
ensure_package("optuna")
ensure_package("kaggle")
ensure_package("dotenv", "python-dotenv")


In [ ]:
import os
import time
import warnings
from datetime import UTC, datetime
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import optuna
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from kaggle.api.kaggle_api_extended import KaggleApi
from scipy.optimize import minimize
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, StratifiedKFold

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)


In [ ]:
# Kaggle + notebook configuration
COMPETITION = "cpe-232-insurance-premium-prediction"
NOTEBOOK_SLUG = "ins232-01-mae-cv-benchmark"

RUN_DOWNLOAD = True
RUN_SUBMISSION = True
FORCE_DOWNLOAD = True

# Balanced defaults
CV_MODE = "kfold"  # 'kfold' | 'stratified'
N_SPLITS = 5
SEED = 42
ENABLE_OPTUNA = False
ENABLE_ABLATIONS = False
ENABLE_ENSEMBLE = True
ENABLE_OPTUNA_FOR = ("catboost", "lightgbm", "xgboost")
OPTUNA_TRIALS = 25
OPTUNA_TIMEOUT_SECONDS = 20 * 60

ID_COL = "id"
TARGET_COL = "Premium Amount"
TEXT_COL = "Customer Feedback"
OPTIONAL_SCHEMA_COLS = {"Policy Start Date"}

MAJOR_MISSING_FIELDS = [
    "Occupation",
    "Marital Status",
    "Location",
    "Credit Score",
    "Number of Dependents",
]

CATEGORICAL_CANDIDATES = [
    "Gender",
    "Marital Status",
    "Education Level",
    "Occupation",
    "Location",
    "Policy Type",
    "Smoking Status",
    "Exercise Frequency",
    "Property Type",
]

RAW_MINIMAL_COLUMNS = [
    "Age",
    "Annual Income",
    "Number of Dependents",
    "Health Score",
    "Previous Claims",
    "Vehicle Age",
    "Credit Score",
    "Insurance Duration",
    "Gender",
    "Marital Status",
    "Education Level",
    "Occupation",
    "Location",
    "Policy Type",
    "Smoking Status",
    "Exercise Frequency",
    "Property Type",
]

PROCESSED_NUMERIC = [
    "Age",
    "Annual Income",
    "Number of Dependents",
    "Health Score",
    "Previous Claims",
    "Vehicle Age",
    "Credit Score",
    "Insurance Duration",
]

LOG_COLS = ["Annual Income", "Health Score", "Credit Score"]
CLIP_COLS = PROCESSED_NUMERIC.copy()

OUTPUT_ROOT = (
    Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
)
ARTIFACT_DIR = OUTPUT_ROOT / "outputs"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Output root:", OUTPUT_ROOT)
print("Artifacts dir:", ARTIFACT_DIR)


In [ ]:
def load_env_from_candidates():
    cwd = Path.cwd().resolve()
    candidates = [cwd / ".env", cwd.parent / ".env", Path("/kaggle/working/.env")]
    loaded = False
    for env_path in candidates:
        if env_path.exists():
            load_dotenv(env_path, override=False)
            print(f"Loaded env from {env_path}")
            loaded = True
            break
    if not loaded:
        print("No .env file found in default candidate locations.")


def resolve_local_data_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "data",
        cwd / "notebooks/2026-03-13-insurance-premium-prediction/data",
        cwd.parent / "data",
    ]
    for c in candidates:
        if c.exists() and (c / "train.csv").exists() and (c / "test.csv").exists():
            return c
    return cwd / "data"


def resolve_kaggle_input_dir(competition: str) -> Path | None:
    direct = Path("/kaggle/input") / competition
    if direct.exists() and (direct / "train.csv").exists():
        return direct

    if Path("/kaggle/input").exists():
        matches = sorted(Path("/kaggle/input").glob(f"*{competition}*"))
        for m in matches:
            if (m / "train.csv").exists() and (m / "test.csv").exists():
                return m
    return None


def ensure_competition_data(
    api: KaggleApi | None, competition: str, run_download: bool, force_download: bool
):
    kaggle_input = resolve_kaggle_input_dir(competition)
    if kaggle_input is not None and not run_download:
        print("Using mounted Kaggle input:", kaggle_input)
        return kaggle_input

    local_data = resolve_local_data_dir()
    local_data.mkdir(parents=True, exist_ok=True)

    train_path = local_data / "train.csv"
    test_path = local_data / "test.csv"
    sample_path = local_data / "sample_submission.csv"

    have_local = train_path.exists() and test_path.exists() and sample_path.exists()

    if run_download:
        if api is None:
            if not have_local:
                raise RuntimeError(
                    "RUN_DOWNLOAD=True but Kaggle auth unavailable and no local data found."
                )
            print("RUN_DOWNLOAD=True but API unavailable. Falling back to local files.")
            return local_data

        if force_download and local_data.exists():
            for z in local_data.glob("*.zip"):
                try:
                    z.unlink()
                except Exception:
                    pass

        if force_download or not have_local:
            print("Downloading competition files via Kaggle API...")
            api.competition_download_files(
                competition, path=str(local_data), force=force_download, quiet=False
            )

            zip_candidates = sorted(local_data.glob("*.zip"))
            if not zip_candidates:
                raise RuntimeError("Kaggle download did not produce a zip file.")
            latest_zip = max(zip_candidates, key=lambda p: p.stat().st_mtime)
            with ZipFile(latest_zip, "r") as zf:
                zf.extractall(local_data)
            print("Extracted:", latest_zip.name)

        have_local = train_path.exists() and test_path.exists() and sample_path.exists()
        if not have_local:
            raise RuntimeError(f"Data missing after download in {local_data}")

    if not have_local:
        raise RuntimeError(
            f"Data not found. Checked local dir {local_data}. "
            "Set RUN_DOWNLOAD=True with valid Kaggle credentials or provide local CSV files."
        )

    return local_data


load_env_from_candidates()
os.environ.pop("KAGGLE_API_TOKEN", None)

api = None
try:
    api = KaggleApi()
    api.authenticate()
    print("Authenticated user:", api.get_config_value("username"))
except Exception as auth_exc:
    print("Kaggle auth unavailable:", auth_exc)

data_dir = ensure_competition_data(api, COMPETITION, RUN_DOWNLOAD, FORCE_DOWNLOAD)
print("Resolved data dir:", data_dir)


In [ ]:
def fail_schema(message: str):
    raise ValueError(f"[SCHEMA ERROR] {message}")


def infer_feature_types(train_df: pd.DataFrame, test_df: pd.DataFrame):
    train_cols = set(train_df.columns)
    test_cols = set(test_df.columns)

    if TARGET_COL not in train_cols:
        fail_schema(f"Missing target column `{TARGET_COL}` in train.csv")
    if TARGET_COL in test_cols:
        fail_schema(f"Target column `{TARGET_COL}` must not exist in test.csv")
    if ID_COL not in train_cols or ID_COL not in test_cols:
        fail_schema(f"Both train and test must contain `{ID_COL}`")

    train_features = train_cols - {TARGET_COL}
    only_train = sorted(train_features - test_cols - OPTIONAL_SCHEMA_COLS)
    only_test = sorted(test_cols - train_features - OPTIONAL_SCHEMA_COLS)

    if only_train or only_test:
        fail_schema(
            f"Train/test feature mismatch. only_train={only_train}, only_test={only_test}"
        )

    common_features = sorted((train_features & test_cols) - {ID_COL})
    if not common_features:
        fail_schema("No common feature columns detected.")

    num_cols = []
    cat_cols = []
    text_cols = []
    for col in common_features:
        s = train_df[col]
        if col == TEXT_COL:
            text_cols.append(col)
            continue
        if pd.api.types.is_numeric_dtype(s):
            num_cols.append(col)
        else:
            avg_len = float(s.astype("string").str.len().fillna(0).mean())
            if avg_len > 40:
                text_cols.append(col)
            else:
                cat_cols.append(col)

    return common_features, num_cols, cat_cols, text_cols


def load_and_validate_data(data_dir: Path):
    train_path = data_dir / "train.csv"
    test_path = data_dir / "test.csv"
    sample_path = data_dir / "sample_submission.csv"

    if not train_path.exists() or not test_path.exists() or not sample_path.exists():
        fail_schema("Expected train.csv, test.csv, sample_submission.csv to exist.")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    sample_df = pd.read_csv(sample_path)

    common_features, num_cols, cat_cols, text_cols = infer_feature_types(
        train_df, test_df
    )

    print("Train shape:", train_df.shape)
    print("Test shape:", test_df.shape)
    print("Common feature count:", len(common_features))
    print("Numeric cols:", num_cols)
    print("Categorical cols:", cat_cols)
    print("Text cols:", text_cols)

    if ID_COL not in sample_df.columns:
        fail_schema(f"sample_submission missing `{ID_COL}`")
    if TARGET_COL not in sample_df.columns:
        fail_schema(f"sample_submission missing `{TARGET_COL}`")

    if not sample_df[ID_COL].isin(test_df[ID_COL]).all():
        print(
            "Warning: sample_submission ids differ from test ids. Submission will use test ids."
        )

    return {
        "train_df": train_df,
        "test_df": test_df,
        "sample_df": sample_df,
        "common_features": common_features,
        "num_cols": num_cols,
        "cat_cols": cat_cols,
        "text_cols": text_cols,
        "train_path": train_path,
        "test_path": test_path,
        "sample_path": sample_path,
    }


ctx = load_and_validate_data(data_dir)
train_df = ctx["train_df"]
test_df = ctx["test_df"]
sample_submission = ctx["sample_df"]
ALL_FEATURES = ctx["common_features"]
NUM_COLS_AUTO = ctx["num_cols"]
CAT_COLS_AUTO = ctx["cat_cols"]
TEXT_COLS_AUTO = ctx["text_cols"]

print("Schema ready.")


In [ ]:
MISSING_TOKENS = {"", "na", "n/a", "none", "null", "nan", "unknown", "missing", "?"}


def normalize_text_like(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    low = s.str.lower()
    s = s.mask(low.isin(MISSING_TOKENS), pd.NA)
    return s


def robust_numeric(series: pd.Series) -> pd.Series:
    out = pd.to_numeric(series, errors="coerce")
    out = out.replace([np.inf, -np.inf], np.nan)
    return out


def clean_base_frames(
    train_raw: pd.DataFrame, test_raw: pd.DataFrame, include_text: bool = True
):
    tr = train_raw.copy(deep=True)
    te = test_raw.copy(deep=True)

    for col in NUM_COLS_AUTO:
        if col in tr.columns:
            tr[col] = robust_numeric(tr[col])
        if col in te.columns:
            te[col] = robust_numeric(te[col])

    tr[TARGET_COL] = robust_numeric(tr[TARGET_COL])

    cat_cols = [c for c in CAT_COLS_AUTO if c in tr.columns]
    for col in cat_cols:
        tr[col] = normalize_text_like(tr[col])
        te[col] = normalize_text_like(te[col])

    if include_text and TEXT_COL in tr.columns:
        tr[TEXT_COL] = tr[TEXT_COL].astype("string").fillna("")
        te[TEXT_COL] = te[TEXT_COL].astype("string").fillna("")
    elif TEXT_COL in tr.columns:
        tr = tr.drop(columns=[TEXT_COL], errors="ignore")
        te = te.drop(columns=[TEXT_COL], errors="ignore")

    return tr, te


def add_missing_indicators(tr: pd.DataFrame, te: pd.DataFrame, columns: list[str]):
    created = []
    for col in columns:
        if col not in tr.columns:
            continue
        if tr[col].isna().mean() <= 0:
            continue
        name = f"mis_{col}"
        tr[name] = tr[col].isna().astype("int8")
        te[name] = te[col].isna().astype("int8")
        created.append(name)
    return created


def fill_numeric_median(tr: pd.DataFrame, te: pd.DataFrame, columns: list[str]):
    for col in columns:
        if col in tr.columns:
            med = float(tr[col].median()) if tr[col].notna().any() else 0.0
            tr[col] = tr[col].fillna(med)
            if col in te.columns:
                te[col] = te[col].fillna(med)


def fill_categorical_missing(tr: pd.DataFrame, te: pd.DataFrame, columns: list[str]):
    for col in columns:
        if col in tr.columns:
            tr[col] = tr[col].fillna("Missing").astype("string")
        if col in te.columns:
            te[col] = te[col].fillna("Missing").astype("string")

        vocab = sorted(set(tr[col].dropna().astype(str).unique()))
        if "Unknown" not in vocab:
            vocab.append("Unknown")
        tr[col] = tr[col].where(tr[col].isin(vocab), "Unknown").astype("category")
        te[col] = te[col].where(te[col].isin(vocab), "Unknown").astype("category")
        te[col] = te[col].cat.set_categories(tr[col].cat.categories)


def add_extended_features(tr: pd.DataFrame, te: pd.DataFrame):
    for col in [c for c in LOG_COLS if c in tr.columns]:
        tr[f"log1p_{col}"] = np.log1p(np.clip(tr[col], a_min=0, a_max=None))
        te[f"log1p_{col}"] = np.log1p(np.clip(te[col], a_min=0, a_max=None))

    for col in [c for c in CLIP_COLS if c in tr.columns]:
        lo = float(tr[col].quantile(0.005))
        hi = float(tr[col].quantile(0.995))
        tr[f"clip_{col}"] = tr[col].clip(lo, hi)
        te[f"clip_{col}"] = te[col].clip(lo, hi)

    if {"Annual Income", "Number of Dependents"}.issubset(set(tr.columns)):
        tr["income_per_dependent"] = tr["Annual Income"] / (
            tr["Number of Dependents"] + 1
        )
        te["income_per_dependent"] = te["Annual Income"] / (
            te["Number of Dependents"] + 1
        )

    if {"Previous Claims", "Insurance Duration"}.issubset(set(tr.columns)):
        tr["claims_per_year"] = (
            tr["Previous Claims"] / tr["Insurance Duration"].replace(0, np.nan)
        ).fillna(0.0)
        te["claims_per_year"] = (
            te["Previous Claims"] / te["Insurance Duration"].replace(0, np.nan)
        ).fillna(0.0)

    if {"Credit Score", "Annual Income"}.issubset(set(tr.columns)):
        tr["credit_income_ratio"] = tr["Credit Score"] / tr["Annual Income"].clip(
            lower=1000.0
        )
        te["credit_income_ratio"] = te["Credit Score"] / te["Annual Income"].clip(
            lower=1000.0
        )

    if "Age" in tr.columns:
        bins = [0, 25, 35, 45, 55, 65, np.inf]
        tr["age_band"] = (
            pd.cut(tr["Age"], bins=bins, include_lowest=True)
            .astype("string")
            .fillna("Missing")
            .astype("category")
        )
        te["age_band"] = (
            pd.cut(te["Age"], bins=bins, include_lowest=True)
            .astype("string")
            .fillna("Missing")
            .astype("category")
        )
        te["age_band"] = te["age_band"].cat.set_categories(
            tr["age_band"].cat.categories
        )

    if "Annual Income" in tr.columns:
        bins = [0, 10000, 30000, 60000, 100000, np.inf]
        tr["income_band"] = (
            pd.cut(tr["Annual Income"], bins=bins, include_lowest=True)
            .astype("string")
            .fillna("Missing")
            .astype("category")
        )
        te["income_band"] = (
            pd.cut(te["Annual Income"], bins=bins, include_lowest=True)
            .astype("string")
            .fillna("Missing")
            .astype("category")
        )
        te["income_band"] = te["income_band"].cat.set_categories(
            tr["income_band"].cat.categories
        )

    if "Previous Claims" in tr.columns:
        bins = [-0.1, 0, 1, 2, 4, 8, np.inf]
        tr["claims_band"] = (
            pd.cut(tr["Previous Claims"], bins=bins, include_lowest=True)
            .astype("string")
            .fillna("Missing")
            .astype("category")
        )
        te["claims_band"] = (
            pd.cut(te["Previous Claims"], bins=bins, include_lowest=True)
            .astype("string")
            .fillna("Missing")
            .astype("category")
        )
        te["claims_band"] = te["claims_band"].cat.set_categories(
            tr["claims_band"].cat.categories
        )


def build_features(
    variant: str,
    train_raw: pd.DataFrame,
    test_raw: pd.DataFrame,
    include_missing_indicators: bool = True,
    include_text: bool = True,
):
    if variant not in {
        "raw_minimal",
        "processed_basic",
        "processed_extended",
        "text_raw",
    }:
        raise ValueError(f"Unknown feature variant: {variant}")

    with_text = include_text or variant == "text_raw"
    tr, te = clean_base_frames(train_raw, test_raw, include_text=with_text)

    if variant == "raw_minimal":
        keep = [c for c in RAW_MINIMAL_COLUMNS if c in tr.columns]
        tr = tr[[ID_COL] + keep + [TARGET_COL]].copy()
        te = te[[ID_COL] + keep].copy()

    feature_cols = [c for c in tr.columns if c not in {ID_COL, TARGET_COL}]

    if variant in {"processed_basic", "processed_extended", "text_raw"}:
        fill_numeric_median(tr, te, [c for c in NUM_COLS_AUTO if c in tr.columns])
        fill_categorical_missing(tr, te, [c for c in CAT_COLS_AUTO if c in tr.columns])
        if include_missing_indicators:
            add_missing_indicators(tr, te, [c for c in feature_cols if c in tr.columns])

    if variant in {"processed_extended", "text_raw"}:
        add_extended_features(tr, te)

    feature_cols = [c for c in tr.columns if c not in {ID_COL, TARGET_COL}]

    if variant != "text_raw" and TEXT_COL in feature_cols:
        tr = tr.drop(columns=[TEXT_COL], errors="ignore")
        te = te.drop(columns=[TEXT_COL], errors="ignore")
        feature_cols = [c for c in feature_cols if c != TEXT_COL]

    for col in feature_cols:
        if pd.api.types.is_numeric_dtype(tr[col]):
            tr[col] = robust_numeric(tr[col]).fillna(0.0)
            te[col] = robust_numeric(te[col]).fillna(0.0)
        elif str(tr[col].dtype) != "category":
            tr[col] = tr[col].astype("string").fillna("Missing").astype("category")
            te[col] = te[col].astype("string").fillna("Missing").astype("category")
            te[col] = te[col].cat.set_categories(tr[col].cat.categories)

    if (
        tr[feature_cols].isna().sum().sum() > 0
        or te[feature_cols].isna().sum().sum() > 0
    ):
        raise ValueError(
            f"Feature variant `{variant}` still has NaNs after processing."
        )

    meta = {
        "feature_cols": feature_cols,
        "cat_cols": [c for c in feature_cols if str(tr[c].dtype) == "category"],
        "text_cols": [c for c in feature_cols if c == TEXT_COL],
        "num_cols": [
            c
            for c in feature_cols
            if c not in [TEXT_COL] and str(tr[c].dtype) != "category"
        ],
    }

    return tr, te, meta


In [ ]:
def make_target_bins(y: pd.Series, n_bins: int = 10):
    # Safe quantile bins for regression stratification fallback to rank bins.
    try:
        b = pd.qcut(y.rank(method="average"), q=n_bins, labels=False, duplicates="drop")
    except Exception:
        b = pd.cut(
            y.rank(method="average"), bins=n_bins, labels=False, include_lowest=True
        )
    return pd.Series(b).fillna(0).astype(int)


def make_folds(y: pd.Series, mode: str, n_splits: int, seed: int):
    idx = np.arange(len(y))
    folds = []
    if mode == "stratified":
        y_bins = make_target_bins(y)
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_idx, va_idx in splitter.split(idx, y_bins):
            folds.append((tr_idx, va_idx))
    elif mode == "kfold":
        splitter = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_idx, va_idx in splitter.split(idx):
            folds.append((tr_idx, va_idx))
    else:
        raise ValueError(f"Unknown CV mode: {mode}")

    if len(folds) != n_splits:
        raise RuntimeError("Failed to construct expected number of folds.")

    fold_id = np.full(len(y), -1, dtype=np.int16)
    for f, (_, va_idx) in enumerate(folds):
        fold_id[va_idx] = f
    if (fold_id < 0).any():
        raise RuntimeError("Some rows were not assigned to folds.")

    return folds, fold_id


def mae_original(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.clip(y_pred, a_min=0.0, a_max=None)
    return float(mean_absolute_error(y_true, y_pred))


def maybe_log_target(y: np.ndarray, mode: str):
    if mode == "raw":
        return y
    if mode == "log1p":
        return np.log1p(np.clip(y, a_min=0.0, a_max=None))
    raise ValueError(f"Unknown target mode: {mode}")


def invert_target(pred: np.ndarray, mode: str):
    if mode == "raw":
        return np.clip(pred, a_min=0.0, a_max=None)
    if mode == "log1p":
        return np.clip(np.expm1(pred), a_min=0.0, a_max=None)
    raise ValueError(f"Unknown target mode: {mode}")


FOLDS, FOLD_ID = make_folds(train_df[TARGET_COL], CV_MODE, N_SPLITS, SEED)
fold_index_path = ARTIFACT_DIR / f"fold_indices_{NOTEBOOK_SLUG}.csv"
pd.DataFrame({ID_COL: train_df[ID_COL], "fold_id": FOLD_ID}).to_csv(
    fold_index_path, index=False
)
print("Saved fold assignments to", fold_index_path)


In [ ]:
def coerce_for_model(
    X: pd.DataFrame, model_family: str, cat_cols: list[str], text_cols: list[str]
):
    Xc = X.copy()

    for col in cat_cols:
        if col in Xc.columns:
            if model_family in {"lightgbm", "catboost"}:
                Xc[col] = Xc[col].astype("category")
            else:
                Xc[col] = (
                    Xc[col]
                    .astype("string")
                    .fillna("Missing")
                    .astype("category")
                    .cat.codes.astype("int32")
                )

    if model_family == "catboost":
        for col in text_cols:
            if col in Xc.columns:
                Xc[col] = Xc[col].astype("string").fillna("")

    if model_family == "xgboost":
        for col in text_cols:
            if col in Xc.columns:
                Xc[col] = Xc[col].astype("string").str.len().fillna(0).astype("float32")
    return Xc


def build_model(
    model_family: str,
    objective_name: str,
    seed: int,
    params_override: dict | None = None,
):
    params_override = params_override or {}

    if model_family == "catboost":
        base = {
            "loss_function": "MAE" if objective_name == "mae" else "Huber:delta=1.0",
            "eval_metric": "MAE",
            "learning_rate": 0.03,
            "depth": 8,
            "l2_leaf_reg": 5.0,
            "iterations": 5000,
            "random_seed": seed,
            "allow_writing_files": False,
            "verbose": False,
        }
        base.update(params_override)
        return CatBoostRegressor(**base)

    if model_family == "lightgbm":
        base = {
            "objective": "regression_l1",
            "metric": "mae",
            "learning_rate": 0.03,
            "n_estimators": 5000,
            "num_leaves": 63,
            "min_child_samples": 30,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "random_state": seed,
            "verbosity": -1,
        }
        base.update(params_override)
        return LGBMRegressor(**base)

    if model_family == "xgboost":
        objective = (
            "reg:absoluteerror" if objective_name == "mae" else "reg:pseudohubererror"
        )
        base = {
            "objective": objective,
            "eval_metric": "mae",
            "learning_rate": 0.03,
            "n_estimators": 5000,
            "max_depth": 8,
            "min_child_weight": 3,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_alpha": 0.1,
            "reg_lambda": 2.0,
            "random_state": seed,
            "tree_method": "hist",
            "enable_categorical": True,
        }
        base.update(params_override)
        return XGBRegressor(**base)

    raise ValueError(f"Unknown model family: {model_family}")


def fit_predict_one_fold(
    model,
    model_family: str,
    X_tr: pd.DataFrame,
    y_tr: np.ndarray,
    X_va: pd.DataFrame,
    y_va: np.ndarray,
    X_te: pd.DataFrame,
    cat_cols: list[str],
    text_cols: list[str],
):
    X_tr_m = coerce_for_model(X_tr, model_family, cat_cols, text_cols)
    X_va_m = coerce_for_model(X_va, model_family, cat_cols, text_cols)
    X_te_m = coerce_for_model(X_te, model_family, cat_cols, text_cols)

    if model_family == "catboost":
        cat_idx = [X_tr_m.columns.get_loc(c) for c in cat_cols if c in X_tr_m.columns]
        text_idx = [X_tr_m.columns.get_loc(c) for c in text_cols if c in X_tr_m.columns]
        model.fit(
            X_tr_m,
            y_tr,
            eval_set=(X_va_m, y_va),
            cat_features=cat_idx if cat_idx else None,
            text_features=text_idx if text_idx else None,
            use_best_model=True,
            early_stopping_rounds=300,
            verbose=False,
        )
    elif model_family == "lightgbm":
        model.fit(
            X_tr_m,
            y_tr,
            eval_set=[(X_va_m, y_va)],
            eval_metric="mae",
            callbacks=[],
        )
    elif model_family == "xgboost":
        model.fit(
            X_tr_m,
            y_tr,
            eval_set=[(X_va_m, y_va)],
            verbose=False,
        )
    else:
        raise ValueError(f"Unknown model family: {model_family}")

    pred_va = model.predict(X_va_m)
    pred_te = model.predict(X_te_m)
    return pred_va, pred_te


def extract_feature_importance(model, model_family: str, feature_names: list[str]):
    try:
        if model_family == "catboost":
            vals = model.get_feature_importance()
            return pd.DataFrame({"feature": feature_names, "importance": vals})
        if model_family == "lightgbm":
            vals = model.feature_importances_
            return pd.DataFrame({"feature": feature_names, "importance": vals})
        if model_family == "xgboost":
            vals = model.feature_importances_
            return pd.DataFrame({"feature": feature_names, "importance": vals})
    except Exception:
        return pd.DataFrame(columns=["feature", "importance"])
    return pd.DataFrame(columns=["feature", "importance"])


In [ ]:
EXP_ROWS = []
OOF_STORE = {}
TEST_STORE = {}
FI_ROWS = []
OPTUNA_HISTORY = {}


def run_experiment(exp_config: dict):
    name = exp_config["name"]
    model_family = exp_config["model_family"]
    objective = exp_config["objective"]
    feature_variant = exp_config["feature_variant"]
    target_mode = exp_config.get("target_mode", "raw")
    include_missing_indicators = exp_config.get("include_missing_indicators", True)
    include_text = exp_config.get("include_text", False)
    params_override = exp_config.get("params_override") or {}
    seed = int(exp_config.get("seed", SEED))

    t0 = time.time()
    tr, te, meta = build_features(
        feature_variant,
        train_df,
        test_df,
        include_missing_indicators=include_missing_indicators,
        include_text=include_text,
    )

    feature_cols = meta["feature_cols"]
    cat_cols = meta["cat_cols"]
    text_cols = meta["text_cols"] if include_text else []

    X = tr[feature_cols].copy()
    X_test = te[feature_cols].copy()
    y_raw = tr[TARGET_COL].to_numpy(dtype=float)
    y_fit = maybe_log_target(y_raw, target_mode)

    oof_pred = np.zeros(len(tr), dtype=float)
    test_fold_preds = []
    fold_maes = []

    for fold, (idx_tr, idx_va) in enumerate(FOLDS):
        model = build_model(
            model_family, objective, seed + fold, params_override=params_override
        )

        X_tr = X.iloc[idx_tr]
        y_tr = y_fit[idx_tr]
        X_va = X.iloc[idx_va]
        y_va_fit = y_fit[idx_va]
        y_va_raw = y_raw[idx_va]

        pred_va_fit, pred_te_fit = fit_predict_one_fold(
            model,
            model_family=model_family,
            X_tr=X_tr,
            y_tr=y_tr,
            X_va=X_va,
            y_va=y_va_fit,
            X_te=X_test,
            cat_cols=cat_cols,
            text_cols=text_cols,
        )

        pred_va = invert_target(pred_va_fit, target_mode)
        pred_te = invert_target(pred_te_fit, target_mode)
        pred_va = np.clip(pred_va, a_min=0.0, a_max=None)
        pred_te = np.clip(pred_te, a_min=0.0, a_max=None)

        oof_pred[idx_va] = pred_va
        test_fold_preds.append(pred_te)
        fold_maes.append(mae_original(y_va_raw, pred_va))

        fi = extract_feature_importance(model, model_family, feature_cols)
        if len(fi):
            fi["experiment"] = name
            fi["fold"] = fold
            FI_ROWS.append(fi)

    test_pred = np.mean(np.vstack(test_fold_preds), axis=0)
    mean_mae = float(np.mean(fold_maes))
    std_mae = float(np.std(fold_maes))
    elapsed = float(time.time() - t0)

    EXP_ROWS.append(
        {
            "experiment": name,
            "model_family": model_family,
            "objective": objective,
            "feature_variant": feature_variant,
            "target_mode": target_mode,
            "include_missing_indicators": int(include_missing_indicators),
            "include_text": int(include_text),
            "seed": seed,
            "fold_mae": "|".join(f"{v:.6f}" for v in fold_maes),
            "cv_mae_mean": mean_mae,
            "cv_mae_std": std_mae,
            "train_time_sec": elapsed,
            "n_features": len(feature_cols),
        }
    )

    OOF_STORE[name] = oof_pred
    TEST_STORE[name] = test_pred

    print(f"[{name}] mean={mean_mae:.6f} std={std_mae:.6f} time={elapsed:.1f}s")
    return {
        "name": name,
        "cv_mae_mean": mean_mae,
        "cv_mae_std": std_mae,
        "fold_maes": fold_maes,
        "oof_pred": oof_pred,
        "test_pred": test_pred,
        "feature_cols": feature_cols,
    }


In [ ]:
def suggest_params(trial: optuna.Trial, model_family: str):
    if model_family == "catboost":
        return {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            "depth": trial.suggest_int("depth", 5, 10),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.0),
            "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
            "iterations": trial.suggest_int("iterations", 1500, 6000),
        }
    if model_family == "lightgbm":
        return {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 255),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 120),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 1000, 6000),
        }
    if model_family == "xgboost":
        return {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            "max_depth": trial.suggest_int("max_depth", 4, 12),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 30.0, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 1000, 6000),
        }
    raise ValueError(f"Unknown model family: {model_family}")


def run_optuna(
    model_family: str,
    feature_variant: str,
    objective: str = "mae",
    target_mode: str = "raw",
    include_text: bool = False,
):
    tr, te, meta = build_features(
        feature_variant,
        train_df,
        test_df,
        include_missing_indicators=True,
        include_text=include_text,
    )
    feature_cols = meta["feature_cols"]
    cat_cols = meta["cat_cols"]
    text_cols = meta["text_cols"] if include_text else []

    X = tr[feature_cols]
    X_test = te[feature_cols]
    y_raw = tr[TARGET_COL].to_numpy(dtype=float)
    y_fit = maybe_log_target(y_raw, target_mode)

    def objective_fn(trial: optuna.Trial):
        params = suggest_params(trial, model_family)
        fold_scores = []

        for fold, (idx_tr, idx_va) in enumerate(FOLDS):
            model = build_model(
                model_family, objective, seed=SEED + fold, params_override=params
            )

            X_tr = X.iloc[idx_tr]
            y_tr = y_fit[idx_tr]
            X_va = X.iloc[idx_va]
            y_va_fit = y_fit[idx_va]
            y_va_raw = y_raw[idx_va]

            pred_va_fit, _ = fit_predict_one_fold(
                model,
                model_family=model_family,
                X_tr=X_tr,
                y_tr=y_tr,
                X_va=X_va,
                y_va=y_va_fit,
                X_te=X_test.iloc[: min(200, len(X_test))],
                cat_cols=cat_cols,
                text_cols=text_cols,
            )
            pred_va = invert_target(pred_va_fit, target_mode)
            fold_scores.append(mae_original(y_va_raw, pred_va))

        return float(np.mean(fold_scores))

    study = optuna.create_study(
        direction="minimize", study_name=f"{NOTEBOOK_SLUG}-{model_family}"
    )
    study.optimize(
        objective_fn,
        n_trials=OPTUNA_TRIALS,
        timeout=OPTUNA_TIMEOUT_SECONDS,
        show_progress_bar=False,
    )

    trials_df = study.trials_dataframe()
    out_path = ARTIFACT_DIR / f"optuna_trials_{model_family}_{NOTEBOOK_SLUG}.csv"
    trials_df.to_csv(out_path, index=False)
    OPTUNA_HISTORY[model_family] = {
        "best_value": float(study.best_value),
        "best_params": study.best_params,
        "trials_path": str(out_path),
    }
    print(f"Optuna[{model_family}] best={study.best_value:.6f} saved={out_path}")
    return study.best_params


In [ ]:
BASE_EXPERIMENTS = [
    {
        "name": "catboost_mae_processed_extended",
        "model_family": "catboost",
        "objective": "mae",
        "feature_variant": "processed_extended",
        "target_mode": "raw",
        "include_text": False,
    },
    {
        "name": "catboost_huber_processed_extended",
        "model_family": "catboost",
        "objective": "huber",
        "feature_variant": "processed_extended",
        "target_mode": "raw",
        "include_text": False,
    },
    {
        "name": "catboost_mae_text_raw",
        "model_family": "catboost",
        "objective": "mae",
        "feature_variant": "text_raw",
        "target_mode": "raw",
        "include_text": True,
    },
    {
        "name": "catboost_mae_processed_extended_logtarget",
        "model_family": "catboost",
        "objective": "mae",
        "feature_variant": "processed_extended",
        "target_mode": "log1p",
        "include_text": False,
    },
    {
        "name": "lightgbm_l1_processed_extended",
        "model_family": "lightgbm",
        "objective": "mae",
        "feature_variant": "processed_extended",
        "target_mode": "raw",
        "include_text": False,
    },
    {
        "name": "xgboost_mae_processed_extended",
        "model_family": "xgboost",
        "objective": "mae",
        "feature_variant": "processed_extended",
        "target_mode": "raw",
        "include_text": False,
    },
    {
        "name": "xgboost_pseudohuber_processed_extended",
        "model_family": "xgboost",
        "objective": "huber",
        "feature_variant": "processed_extended",
        "target_mode": "raw",
        "include_text": False,
    },
]


if ENABLE_OPTUNA:
    for fam in ENABLE_OPTUNA_FOR:
        if fam == "catboost":
            best = run_optuna("catboost", "processed_extended", objective="mae")
            for exp in BASE_EXPERIMENTS:
                if exp["model_family"] == "catboost" and exp["objective"] == "mae":
                    exp["params_override"] = best
        elif fam == "lightgbm":
            best = run_optuna("lightgbm", "processed_extended", objective="mae")
            for exp in BASE_EXPERIMENTS:
                if exp["model_family"] == "lightgbm":
                    exp["params_override"] = best
        elif fam == "xgboost":
            best = run_optuna("xgboost", "processed_extended", objective="mae")
            for exp in BASE_EXPERIMENTS:
                if exp["model_family"] == "xgboost" and exp["objective"] == "mae":
                    exp["params_override"] = best

for exp in BASE_EXPERIMENTS:
    run_experiment(exp)

leaderboard_df = (
    pd.DataFrame(EXP_ROWS)
    .sort_values("cv_mae_mean", ascending=True)
    .reset_index(drop=True)
)
display(leaderboard_df)


In [ ]:
def run_ablation_suite():
    ablations = [
        {
            "name": "ablation_raw_minimal_catboost_mae",
            "model_family": "catboost",
            "objective": "mae",
            "feature_variant": "raw_minimal",
            "target_mode": "raw",
            "include_missing_indicators": False,
            "include_text": False,
        },
        {
            "name": "ablation_processed_basic_catboost_mae",
            "model_family": "catboost",
            "objective": "mae",
            "feature_variant": "processed_basic",
            "target_mode": "raw",
            "include_missing_indicators": True,
            "include_text": False,
        },
        {
            "name": "ablation_processed_extended_catboost_mae",
            "model_family": "catboost",
            "objective": "mae",
            "feature_variant": "processed_extended",
            "target_mode": "raw",
            "include_missing_indicators": True,
            "include_text": False,
        },
        {
            "name": "ablation_no_missing_indicators",
            "model_family": "catboost",
            "objective": "mae",
            "feature_variant": "processed_extended",
            "target_mode": "raw",
            "include_missing_indicators": False,
            "include_text": False,
        },
        {
            "name": "ablation_with_text",
            "model_family": "catboost",
            "objective": "mae",
            "feature_variant": "text_raw",
            "target_mode": "raw",
            "include_missing_indicators": True,
            "include_text": True,
        },
        {
            "name": "ablation_log_target",
            "model_family": "catboost",
            "objective": "mae",
            "feature_variant": "processed_extended",
            "target_mode": "log1p",
            "include_missing_indicators": True,
            "include_text": False,
        },
        {
            "name": "ablation_huber_objective",
            "model_family": "catboost",
            "objective": "huber",
            "feature_variant": "processed_extended",
            "target_mode": "raw",
            "include_missing_indicators": True,
            "include_text": False,
        },
    ]
    for exp in ablations:
        if exp["name"] in set(pd.DataFrame(EXP_ROWS)["experiment"]):
            continue
        run_experiment(exp)


if ENABLE_ABLATIONS:
    run_ablation_suite()
    leaderboard_df = (
        pd.DataFrame(EXP_ROWS)
        .sort_values("cv_mae_mean", ascending=True)
        .reset_index(drop=True)
    )
    display(leaderboard_df)


In [ ]:
def analyze_errors(oof_df: pd.DataFrame):
    rows = []

    slice_cols = ["Policy Type", "Smoking Status", "Occupation"]
    for col in slice_cols:
        if col not in oof_df.columns:
            continue
        grp = (
            oof_df.groupby(col, dropna=False)
            .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))
            .reset_index(name="mae")
        )
        grp["slice_type"] = col
        rows.append(grp.rename(columns={col: "slice_value"}))

    if "Age" in oof_df.columns:
        age_band = pd.cut(
            oof_df["Age"], bins=[0, 25, 35, 45, 55, 65, np.inf], include_lowest=True
        )
        grp = (
            oof_df.assign(age_band=age_band)
            .groupby("age_band")
            .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))
            .reset_index(name="mae")
        )
        grp["slice_type"] = "age_band"
        rows.append(grp.rename(columns={"age_band": "slice_value"}))

    if "Annual Income" in oof_df.columns:
        income_band = pd.cut(
            oof_df["Annual Income"],
            bins=[0, 10000, 30000, 60000, 100000, np.inf],
            include_lowest=True,
        )
        grp = (
            oof_df.assign(income_band=income_band)
            .groupby("income_band")
            .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))
            .reset_index(name="mae")
        )
        grp["slice_type"] = "income_band"
        rows.append(grp.rename(columns={"income_band": "slice_value"}))

    if "Previous Claims" in oof_df.columns:
        claims_band = pd.cut(
            oof_df["Previous Claims"],
            bins=[-0.1, 0, 1, 2, 4, 8, np.inf],
            include_lowest=True,
        )
        grp = (
            oof_df.assign(claims_band=claims_band)
            .groupby("claims_band")
            .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))
            .reset_index(name="mae")
        )
        grp["slice_type"] = "claims_band"
        rows.append(grp.rename(columns={"claims_band": "slice_value"}))

    for col in MAJOR_MISSING_FIELDS:
        if col not in oof_df.columns:
            continue
        miss = oof_df[col].isna() | (
            oof_df[col].astype("string").str.lower().isin(MISSING_TOKENS)
        )
        tmp = oof_df.assign(_missing=np.where(miss, "missing", "non_missing"))
        grp = (
            tmp.groupby("_missing")
            .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))
            .reset_index(name="mae")
        )
        grp["slice_type"] = f"missing_{col}"
        rows.append(grp.rename(columns={"_missing": "slice_value"}))

    if not rows:
        return pd.DataFrame(columns=["slice_type", "slice_value", "mae"])

    out = pd.concat(rows, ignore_index=True)
    out = out[["slice_type", "slice_value", "mae"]]
    out = out.sort_values(["slice_type", "mae"], ascending=[True, True])
    return out


def inverse_mae_weights(results_df: pd.DataFrame, model_names: list[str]):
    eps = 1e-8
    inv = []
    for n in model_names:
        mae = float(
            results_df.loc[results_df["experiment"] == n, "cv_mae_mean"].iloc[0]
        )
        inv.append(1.0 / (mae + eps))
    w = np.array(inv, dtype=float)
    w = w / w.sum()
    return w


def optimize_blend_weights(y_true: np.ndarray, pred_matrix: np.ndarray):
    n = pred_matrix.shape[1]
    x0 = np.ones(n) / n

    cons = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)
    bounds = [(0.0, 1.0)] * n

    def obj(w):
        pred = pred_matrix @ w
        return mean_absolute_error(y_true, np.clip(pred, 0.0, None))

    res = minimize(obj, x0=x0, bounds=bounds, constraints=cons, method="SLSQP")
    if not res.success:
        return x0
    return np.clip(res.x, 0.0, 1.0)


def build_ensembles(results_df: pd.DataFrame, oof_store: dict, test_store: dict):
    if len(results_df) < 2:
        return []

    top = results_df.sort_values("cv_mae_mean").head(min(4, len(results_df)))
    model_names = top["experiment"].tolist()

    y_true = train_df[TARGET_COL].to_numpy(dtype=float)
    oof_mat = np.column_stack([oof_store[n] for n in model_names])
    test_mat = np.column_stack([test_store[n] for n in model_names])

    outputs = []

    simple_oof = np.mean(oof_mat, axis=1)
    simple_test = np.mean(test_mat, axis=1)
    simple_name = f"ensemble_simple_top{len(model_names)}"
    OOF_STORE[simple_name] = np.clip(simple_oof, 0.0, None)
    TEST_STORE[simple_name] = np.clip(simple_test, 0.0, None)
    outputs.append(simple_name)

    inv_w = inverse_mae_weights(results_df, model_names)
    inv_oof = oof_mat @ inv_w
    inv_test = test_mat @ inv_w
    inv_name = f"ensemble_inverse_mae_top{len(model_names)}"
    OOF_STORE[inv_name] = np.clip(inv_oof, 0.0, None)
    TEST_STORE[inv_name] = np.clip(inv_test, 0.0, None)
    outputs.append(inv_name)

    opt_w = optimize_blend_weights(y_true, oof_mat)
    opt_oof = oof_mat @ opt_w
    opt_test = test_mat @ opt_w
    opt_name = f"ensemble_optimized_oof_top{len(model_names)}"
    OOF_STORE[opt_name] = np.clip(opt_oof, 0.0, None)
    TEST_STORE[opt_name] = np.clip(opt_test, 0.0, None)
    outputs.append(opt_name)

    for n in outputs:
        pred = OOF_STORE[n]
        fold_maes = []
        for _, idx_va in FOLDS:
            fold_maes.append(mae_original(y_true[idx_va], pred[idx_va]))
        EXP_ROWS.append(
            {
                "experiment": n,
                "model_family": "ensemble",
                "objective": "blend",
                "feature_variant": "na",
                "target_mode": "raw",
                "include_missing_indicators": 1,
                "include_text": 0,
                "seed": SEED,
                "fold_mae": "|".join(f"{v:.6f}" for v in fold_maes),
                "cv_mae_mean": float(np.mean(fold_maes)),
                "cv_mae_std": float(np.std(fold_maes)),
                "train_time_sec": 0.0,
                "n_features": 0,
            }
        )

    return outputs


In [ ]:
results_df = pd.DataFrame(EXP_ROWS).sort_values("cv_mae_mean").reset_index(drop=True)

if ENABLE_ENSEMBLE:
    _ = build_ensembles(results_df, OOF_STORE, TEST_STORE)
    results_df = (
        pd.DataFrame(EXP_ROWS).sort_values("cv_mae_mean").reset_index(drop=True)
    )

display(results_df)

exp_path = ARTIFACT_DIR / f"experiments_{NOTEBOOK_SLUG}.csv"
results_df.to_csv(exp_path, index=False)

best_exp = results_df.iloc[0]["experiment"]
best_oof = OOF_STORE[best_exp]
best_test = TEST_STORE[best_exp]

# OOF exports
base_oof = pd.DataFrame(
    {
        ID_COL: train_df[ID_COL],
        "y_true": train_df[TARGET_COL],
        "y_pred": best_oof,
        "abs_error": np.abs(train_df[TARGET_COL].to_numpy(dtype=float) - best_oof),
        "experiment": best_exp,
    }
)

# Attach common slicing columns when available
for c in [
    "Policy Type",
    "Smoking Status",
    "Occupation",
    "Age",
    "Annual Income",
    "Previous Claims",
] + MAJOR_MISSING_FIELDS:
    if c in train_df.columns and c not in base_oof.columns:
        base_oof[c] = train_df[c]

oof_path = ARTIFACT_DIR / f"oof_predictions_{NOTEBOOK_SLUG}.csv"
base_oof.to_csv(oof_path, index=False)

# Test prediction table with all model predictions
test_pred_table = pd.DataFrame({ID_COL: test_df[ID_COL]})
for name, pred in TEST_STORE.items():
    test_pred_table[name] = np.clip(pred, 0.0, None)

test_pred_path = ARTIFACT_DIR / f"test_predictions_{NOTEBOOK_SLUG}.csv"
test_pred_table.to_csv(test_pred_path, index=False)

# Feature importance (when available)
if FI_ROWS:
    fi_df = pd.concat(FI_ROWS, ignore_index=True)
    fi_summary = (
        fi_df.groupby(["experiment", "feature"], as_index=False)["importance"]
        .mean()
        .sort_values(["experiment", "importance"], ascending=[True, False])
    )
else:
    fi_summary = pd.DataFrame(columns=["experiment", "feature", "importance"])

fi_path = ARTIFACT_DIR / f"feature_importance_{NOTEBOOK_SLUG}.csv"
fi_summary.to_csv(fi_path, index=False)

# Error analysis for best model
error_slice_df = analyze_errors(base_oof)
error_slice_path = ARTIFACT_DIR / f"error_slices_{NOTEBOOK_SLUG}.csv"
error_slice_df.to_csv(error_slice_path, index=False)

# Submission
submission_df = pd.DataFrame(
    {
        ID_COL: test_df[ID_COL],
        TARGET_COL: np.clip(best_test, 0.0, None),
    }
)

sub_path = OUTPUT_ROOT / f"submission_{NOTEBOOK_SLUG}.csv"
submission_df.to_csv(sub_path, index=False)

print("Saved experiments:", exp_path)
print("Saved OOF predictions:", oof_path)
print("Saved test predictions:", test_pred_path)
print("Saved feature importance:", fi_path)
print("Saved error slices:", error_slice_path)
print("Saved submission:", sub_path)

# Simple stability + recommendation
best_row = results_df.iloc[0]
stable = best_row["cv_mae_std"] <= results_df["cv_mae_std"].median()
print("\nRecommended model:", best_exp)
print("CV MAE mean:", round(float(best_row["cv_mae_mean"]), 6))
print("CV MAE std :", round(float(best_row["cv_mae_std"]), 6))
print("Stable across folds:", bool(stable))


In [ ]:
if RUN_SUBMISSION:
    if api is None:
        raise RuntimeError("RUN_SUBMISSION=True but Kaggle API auth is not available")

    submit_message = (
        f"{NOTEBOOK_SLUG} | "
        f"cv_mode={CV_MODE} n_splits={N_SPLITS} seed={SEED} | "
        f"time={datetime.now(UTC).isoformat()}"
    )
    api.competition_submit(
        file_name=str(OUTPUT_ROOT / f"submission_{NOTEBOOK_SLUG}.csv"),
        message=submit_message,
        competition=COMPETITION,
    )
    print("Submission sent:", submit_message)
